<div style="display:flex; align-items:center; gap:18px; text-align:left">
  <img src="https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/assets/logo_ust.png" width="100">
  <div>
    <p>Diplomado en Inteligencia Artificial para los Negocios</p>
    <p>Facultad de Ingeniería y Negocios</p>
    <p>Módulo 2: Fundamentos de Machine Learning y herramientas Low Code</p>
    <p>Semana 03: Algoritmos de Regresión</p>
  </div>
</div>

# 03 · Algoritmos de Regresión

En la Clase 2 dejamos los **datos listos**. Hoy entrenamos modelos para **predecir un número** —el precio de venta de una casa (`precio_uf`)— y, sobre todo, aprendemos a **leer sus métricas, interpretarlos y decidir** cuál sirve.

El flujo (con PyCaret, *low-code*): **preparar → comparar modelos → elegir el mejor → mirarlo por dentro (residuos e importancia) → evaluar en el conjunto de prueba**. El foco no es escribir código, sino **interpretar y decidir**.

## Preparación del entorno

Primero instalamos las librerías necesarias para ejecutar el notebook en Colab o en local. La salida queda oculta para no llenar la pantalla con texto técnico.


In [1]:
%%capture
!pip install -q pycaret==4.0.0a8 plotly ipywidgets xgboost catboost lightgbm "setuptools<81"


## Setup técnico

Esta celda reúne importaciones y configuración general. No es contenido propio de regresión; solo deja listo el entorno para que las celdas importantes se concentren en interpretar modelos.


In [2]:
%matplotlib inline
import os, warnings
warnings.filterwarnings('ignore')

import plotly.express as px
from IPython.display import HTML
from sklearn import set_config
from sklearn.utils import estimator_html_repr
from IPython.utils import io

# evita que Jupyter/GitHub muestre el pipeline como HTML enorme.
set_config(display='text')

## Cargar datos

Usaremos el mismo dataset de casas de la Clase 2, ahora con el foco puesto en entrenar y evaluar modelos de regresión.


In [3]:
import pandas as pd

REPO = 'https://raw.githubusercontent.com/sebastiancontz/ust-diplomado-ia-modulo-ml/main/modulos/modulo-2-fundamentos-ml/2026/datasets/'
BASE = '../datasets/' if os.path.exists('../datasets') else REPO

df = pd.read_csv(BASE + 'precios_casas_rm.csv')
print('Filas y columnas:', df.shape)
df.head()


Filas y columnas: (2377, 15)


,latitud,longitud,area_construida,area_total,dormitorios,banos,estacionamiento,bodega,piscina,quincho,precio_uf,comuna,distancia_a_estacion_cercana,estacion_cercana,metro
0,-33.286041,-70.894675,50.0,120.0,3,1,No,No,No,No,1010.260,LAMPA,21898.81094,No metro,No
1,-33.280238,-70.885977,103.0,243.0,3,3,No,No,No,No,827.240,LAMPA,21316.06967,No metro,No
2,-33.280178,-70.885673,46.0,166.0,3,2,No,No,No,No,1144.000,LAMPA,21290.61472,No metro,No
3,-33.279629,-70.885345,103.0,90.0,5,2,Si,Si,Si,Si,741.235,LAMPA,21282.65947,No metro,No
4,-33.280996,-70.882782,70.0,90.0,3,1,Si,Si,Si,Si,888.615,LAMPA,20992.96901,No metro,No


## El problema: predecir un número

El **target** (lo que queremos predecir) es `precio_uf`, una variable **numérica continua** → es un problema de **regresión**. El resto de las columnas son las **features** (área, comuna, baños, cercanía a metro…). Veamos rápido la relación entre la superficie y el precio:

In [4]:
df_plot = df.assign(
    dormitorios_cat=df['dormitorios'].round().astype('Int64').astype('string').fillna('sin dato')
)

px.scatter(
    df_plot,
    x='area_construida',
    y='precio_uf',
    color='dormitorios_cat',
    hover_data=['comuna', 'banos'],
    title='Precio vs. superficie construida',
    labels={
        'area_construida': 'área construida (m²)',
        'precio_uf': 'precio (UF)',
        'dormitorios_cat': 'dormitorios'
    },
)


Se ve una **tendencia** (a mayor superficie, mayor precio) con bastante dispersión: ninguna recta pasará por todos los puntos. Un buen modelo **captura la tendencia**, no memoriza cada caso.

## 1. Preparar el experimento

Con PyCaret, preparar los datos para modelar es **una línea**. Al crear el `RegressionExperiment` y llamar `.fit(df)`, PyCaret automáticamente **separa entrenamiento y prueba**, **codifica** las categóricas (las traduce a números) e **imputa** los faltantes. El `session_id` fija una semilla para que el resultado sea **reproducible**.

En este notebook hacemos explícito `train_size=0.70`: usamos cerca del **70%** de los datos para entrenar y dejamos cerca del **30%** como conjunto de prueba. Esa división 70/30 es una práctica común cuando el dataset es de tamaño moderado; en industria también verás 80/20 o 75/25. Lo importante no es memorizar un porcentaje, sino **separar datos nuevos antes de entrenar** y mantener esa regla consistente.

<p align="center"></p>


In [5]:
from pycaret.tasks import RegressionExperiment

TRAIN_SIZE = 0.70
FOLDS = 5

exp = RegressionExperiment(
    target='precio_uf',
    session_id=42,
    train_size=TRAIN_SIZE,
    fold=FOLDS,
).fit(df)

# ¿cuántas filas quedaron para entrenar y cuántas para el 'examen final'?
print(f'Train size configurado: {TRAIN_SIZE:.0%}')
print(f'Folds de validación cruzada: {FOLDS}')
print('Entrenamiento:', exp.X_train.shape, '| Prueba (holdout):', exp.X_test.shape)


Train size configurado: 70%
Folds de validación cruzada: 5
Entrenamiento: (1663, 14) | Prueba (holdout): (714, 14)


El **conjunto de prueba** (holdout) queda apartado: el modelo **no lo verá** al entrenar. Lo usaremos **una sola vez**, al final, como examen con datos nuevos.

Además configuramos `fold=5`: durante la comparación, PyCaret divide el entrenamiento en 5 partes, entrena varias veces y promedia. Eso entrega una lectura más estable que una única partición. En clasificación veremos una variante especial, `StratifiedKFold`, porque ahí importa mantener la proporción de clases en cada fold.


## 2. Comparar modelos

Ahora la parte central: PyCaret entrena una **galería de modelos** y los **ordena por desempeño** en una tabla (*leaderboard*). Para esta clase comparamos las familias que vimos: **regresión lineal** (`lr`, la línea base), **árbol de decisión** (`dt`) y ensambles: *random forest* (`rf`), *gradient boosting* (`gbr`) y tres implementaciones muy usadas en datos tabulares: **XGBoost**, **CatBoost** y **LightGBM**.

PyCaret no evalúa con los datos de entrenamiento directamente: usa **validación cruzada** (*cross-validation*), que rota el examen entre varias particiones y promedia. Por eso el ranking es **más confiable** que una sola división.

<p align="center"></p>


In [6]:
# include limita la galería a las familias de la clase (más rápido).
# errors='ignore' permite seguir si una librería opcional falla en Colab.
modelos_clase = ['lr', 'dt', 'rf', 'gbr', 'xgboost', 'catboost', 'lightgbm']

# LightGBM puede imprimir muchos logs internos; los ocultamos sin esconder la tabla final.
with io.capture_output():
    res = exp.compare_models(include=modelos_clase, errors='ignore')
leaderboard = res.leaderboard
leaderboard


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,catboost,598.6744,8.458962e+05,917.0965,0.9338,0.2324,0.1783
1,lightgbm,658.3046,1.003275e+06,998.7253,0.9214,0.2554,0.1978
2,xgboost,637.6693,1.076231e+06,1034.2835,0.9158,0.2669,0.1964
3,rf,696.9419,1.198602e+06,1091.9837,0.9063,0.2795,0.2229
4,gbr,778.7094,1.235128e+06,1108.6040,0.9032,0.2856,0.2354
5,dt,885.1403,2.400975e+06,1540.9709,0.8121,0.3807,0.2653
6,lr,1545.7116,4.138293e+06,2032.8635,0.6770,0.6127,0.5353


### Cómo leer el *leaderboard*

Cada fila es un modelo; cada columna, una **métrica** (promedio de la validación cruzada):

- **MAE** y **RMSE** están en **UF** (mientras más bajos, mejor). El MAE es el error promedio; el RMSE **castiga más los errores grandes**.
- **R²** es ≤ 1 (1 = casi perfecto; 0 = no mejora al promedio; negativo = peor que el promedio): qué proporción de la variación del precio explica el modelo. Más cerca de 1, mejor, pero **no lo mires solo**.
- **MAPE** aparece como proporción: `0.20` significa cerca de **20%** de error promedio relativo.

Lo típico en datos tabulares es que la **línea base** (`lr`) sea fácil de explicar, los **árboles** capturen reglas no lineales y los **ensambles** (`rf`, `gbr`, XGBoost, CatBoost, LightGBM) compitan fuerte. El ganador exacto puede cambiar por dataset y métrica; por eso no elegimos por nombre del algoritmo, sino por **desempeño, interpretabilidad y uso de negocio**.

<p align="center"></p>


## 3. Comparar con y sin remoción automática de outliers

En la Clase 2 miramos outliers con criterio humano. PyCaret también permite probar una versión automática con `remove_outliers=True`. La idea es entrenar otro experimento donde PyCaret detecta observaciones anómalas en el **conjunto de entrenamiento** y compara modelos sin esos casos extremos.

Esto no significa que siempre debamos borrar outliers. La pregunta correcta es de negocio: **¿son errores de dato o casos reales importantes?** Si son casas de lujo reales, quitarlas puede mejorar una métrica promedio, pero empobrecer el modelo para ese segmento.


Antes de decidir, **miremos si hay outliers** con un *boxplot* del precio (como en la Clase 2). Los puntos más allá de los bigotes son casas con precios muy por encima del resto: candidatas a outliers que conviene **investigar**, no borrar a ciegas.

In [7]:
px.box(
    df,
    x='precio_uf',
    points='outliers',
    title='¿Hay outliers en el precio? (boxplot)',
    labels={'precio_uf': 'precio (UF)'},
)

In [8]:
exp_sin_outliers = RegressionExperiment(
    target='precio_uf',
    session_id=42,
    train_size=TRAIN_SIZE,
    fold=FOLDS,
    remove_outliers=True,
).fit(df)

# Misma comparación, pero silenciando logs internos de LightGBM.
with io.capture_output():
    res_sin_outliers = exp_sin_outliers.compare_models(include=modelos_clase, errors='ignore')
leaderboard_sin_outliers = res_sin_outliers.leaderboard

metricas_resumen = ['MAE', 'RMSE', 'R2', 'MAPE']


def resumen_leaderboard(tabla, experimento, top=5):
    resumen = tabla.copy().head(top)
    if 'Model' in resumen.columns:
        resumen = resumen.rename(columns={'Model': 'modelo'})
    elif 'modelo' not in resumen.columns:
        resumen = resumen.reset_index().rename(columns={resumen.index.name or 'index': 'modelo'})

    columnas = ['modelo'] + metricas_resumen
    return resumen[columnas].assign(experimento=experimento)


comparacion_outliers = pd.concat(
    [
        resumen_leaderboard(leaderboard, 'base'),
        resumen_leaderboard(leaderboard_sin_outliers, 'remove_outliers=True'),
    ],
    ignore_index=True,
)
comparacion_outliers


,modelo,MAE,RMSE,R2,MAPE,experimento
0,catboost,598.6744,917.0965,0.9338,0.1783,base
1,lightgbm,658.3046,998.7253,0.9214,0.1978,base
2,xgboost,637.6693,1034.2835,0.9158,0.1964,base
3,rf,696.9419,1091.9837,0.9063,0.2229,base
4,gbr,778.7094,1108.6040,0.9032,0.2354,base
5,catboost,595.3106,911.4491,0.9353,0.1702,remove_outliers=True
6,lightgbm,643.7409,992.9928,0.9231,0.1829,remove_outliers=True
7,xgboost,628.2098,1023.3366,0.9182,0.1819,remove_outliers=True
8,rf,661.8523,1025.4024,0.9179,0.1992,remove_outliers=True
9,gbr,765.6559,1108.4184,0.9041,0.2259,remove_outliers=True


Lee la tabla como una prueba de sensibilidad: si el ranking cambia mucho, el dataset depende bastante de casos extremos. Para el resto del notebook seguimos con el **experimento base**, porque queremos evaluar el problema real completo y no esconder casos caros que podrían existir en producción.


## 4. El mejor modelo

PyCaret deja el ganador a mano en `.best`. Es un *pipeline* completo: incluye la preparación de los datos y el modelo final. Aquí lo mostramos con la representación HTML de scikit-learn para poder abrir y cerrar sus pasos. Si tu prioridad de negocio fuera otra métrica, podrías ordenar por ella, por ejemplo `exp.compare_models(sort='MAE')`.


In [9]:
mejor = res.best

# representación HTML interactiva del pipeline de scikit-learn.
HTML(estimator_html_repr(mejor))

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('catboost', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](0,)",[]
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](14,)","['latitud','longitud','area_construida',...,'distancia_a_estacion_cercana', 'estacion_cercana','metro']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,14
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical_pipeline', ...), ('categorical_pipeline', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transforme

## 5. Mirar el modelo por dentro

Un buen número no basta: conviene entender **dónde se equivoca** y **qué variables usa**.

### Residuos (dónde se equivoca)

Un **residuo** es la diferencia entre el valor real y la predicción. Si los residuos muestran patrones, el modelo probablemente está dejando algo importante sin capturar.


In [10]:
exp.plot_model(mejor, plot='residuals')

### Importancia de variables (qué pesa)

Qué variables mueven más la predicción. Suele confirmar lo esperable (área, comuna) y a veces revela algo no obvio. **Ojo:** que una variable *pese* no significa que *cause* el precio (asociación, no causalidad).

In [11]:
exp.plot_model(mejor, plot='feature')

## 6. El examen final: el conjunto de prueba

La validación cruzada nos sirvió para **elegir**. Ahora evaluamos el modelo elegido **una sola vez** sobre el **conjunto de prueba** (datos que nunca vio): es el examen más honesto.


In [12]:
pred = exp.predict_model(mejor)   # predice sobre el holdout
pred.metrics                      # métricas sobre datos NO vistos

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,CatBoostRegressor,649.6005,983721.0894,991.8271,0.919,0.295,0.2355


Compara estas métricas con las del *leaderboard*: si son **parecidas**, el modelo **generaliza** bien. Si en el examen final empeoran mucho, había **sobreajuste**.

Y la lectura de negocio: un **MAE** de, digamos, ~650 UF significa que, en promedio, la tasación se equivoca por ~650 UF. ¿Es aceptable? Depende del caso: trivial para una casa de 10.000 UF, relevante para una de 1.500 UF. **La métrica se traduce en una decisión.**

In [13]:
# las predicciones, casa por casa (columna prediction_label = la predicción)
predicciones = pred.predictions.copy()
predicciones[['area_construida', 'comuna', 'precio_uf', 'prediction_label']].head(10)


,area_construida,comuna,precio_uf,prediction_label
2055,137.0,LA REINA,9190.000000,9738.117820
1813,110.0,LA REINA,6123.000000,7491.206889
100,NaN,COLINA,10270.000000,5417.634240
1349,63.0,LA FLORIDA,2289.999989,1150.755685
56,107.0,COLINA,7900.000000,8757.167575
1743,56.0,PUENTE ALTO,997.480000,1217.262282
1307,64.0,PEDRO AGUIRRE CERDA,1884.940000,1643.701740
611,100.0,RENCA,3342.900000,2831.779929
1058,121.0,PENALOLEN,9990.000000,8530.188701
1995,130.0,SAN MIGUEL,6731.920000,5076.974209


El gráfico siguiente compara **precio real vs. precio predicho**. La línea diagonal representa predicción perfecta: mientras más cerca esté un punto de esa línea, menor es el error. Los puntos más alejados son buenos candidatos para discutir límites del modelo.

**Prompt para pedirle ayuda a una IA:**

```text
Tengo un DataFrame llamado predicciones con estas columnas: precio_uf, prediction_label, comuna, area_construida, dormitorios y banos.
Quiero un gráfico interactivo con Plotly Express para comparar precio real vs. precio predicho.
Usa precio_uf en el eje x y prediction_label en el eje y.
Crea una columna error_abs_uf con el error absoluto en UF y úsala como color.
Agrega al hover comuna, area_construida, dormitorios y banos.
Agrega una línea diagonal gris punteada que represente predicción perfecta.
Usa títulos y etiquetas en español, con ancho 800 y alto 520.
Devuélveme solo el código Python.
```


In [14]:
predicciones['error_abs_uf'] = (
    predicciones['precio_uf'] - predicciones['prediction_label']
).abs()

min_val = min(predicciones['precio_uf'].min(), predicciones['prediction_label'].min())
max_val = max(predicciones['precio_uf'].max(), predicciones['prediction_label'].max())

fig = px.scatter(
    predicciones,
    x='precio_uf',
    y='prediction_label',
    color='error_abs_uf',
    hover_data=['comuna', 'area_construida', 'dormitorios', 'banos'],
    title='Precio real vs. precio predicho',
    labels={
        'precio_uf': 'precio real (UF)',
        'prediction_label': 'precio predicho (UF)',
        'error_abs_uf': 'error absoluto (UF)'
    },
)
fig.add_shape(
    type='line',
    x0=min_val,
    y0=min_val,
    x1=max_val,
    y1=max_val,
    line=dict(color='gray', dash='dash'),
)
fig.update_layout(width=800, height=520)
fig.show()


### Métricas por comuna

El promedio global puede esconder diferencias por segmento. Revisemos el desempeño del **mejor modelo del leaderboard** por comuna: MAE y RMSE quedan en UF; R² se calcula solo cuando hay al menos dos observaciones en esa comuna, porque con un solo caso no es informativo.

**Prompt para pedirle ayuda a una IA:**

```text
Tengo un DataFrame llamado predicciones con las columnas comuna, precio_uf y prediction_label.
Quiero una tabla de métricas por comuna para evaluar un modelo de regresión.
Para cada comuna calcula: cantidad de casos, MAE, RMSE y R2.
MAE debe ser el promedio del error absoluto entre precio_uf y prediction_label.
RMSE debe ser la raíz del promedio de los errores al cuadrado.
R2 debe calcularse con r2_score de sklearn.metrics, pero solo si la comuna tiene al menos dos observaciones; si tiene una sola, deja R2 como pd.NA.
Ordena la tabla desde las comunas con mayor MAE hacia las de menor MAE.
Devuélveme solo el código Python.
```


In [15]:
from sklearn.metrics import r2_score

filas = []
for comuna, grupo in predicciones.groupby('comuna'):
    error = grupo['precio_uf'] - grupo['prediction_label']
    filas.append({
        'comuna': comuna,
        'casos': len(grupo),
        'MAE': error.abs().mean(),
        'RMSE': (error.pow(2).mean()) ** 0.5,
        'R2': r2_score(grupo['precio_uf'], grupo['prediction_label']) if len(grupo) >= 2 else pd.NA,
    })

metricas_por_comuna = (
    pd.DataFrame(filas)
    .sort_values(['MAE', 'casos'], ascending=[False, False])
    .reset_index(drop=True)
)
metricas_por_comuna


,comuna,casos,MAE,RMSE,R2
0,PROVIDENCIA,7,1723.178245,2028.014921,-0.611160
1,LO PRADO,4,1472.491055,1848.558283,-0.436255
2,SANTIAGO,11,1423.097943,1675.365024,0.437565
3,LAS CONDES,36,1266.097712,1827.930711,0.694191
4,VITACURA,9,1251.556653,1416.902540,0.219662
5,MACUL,16,1225.031622,1801.485795,0.083636
6,COLINA,30,1175.170364,1621.304158,0.761029
7,NUNOA,28,1123.508364,1498.310799,0.695618
8,CONCHALI,7,1018.089806,1201.476302,0.083834
9,QUINTA NORMAL,11,993.687146,1219.675598,0.756584


Si ves algún `NaN` en `area_construida` arriba, es normal: el dataset trae faltantes y **PyCaret los imputó automáticamente** al preparar los datos (lo que hicimos en la Clase 2). El modelo igual predice.

## 7. Ahora tú

Prueba y **observa cómo cambian las decisiones** (las soluciones se publican después):

1. Vuelve a comparar ordenando por otra métrica: `exp.compare_models(sort='MAE')`. ¿Cambia el ganador? ¿Por qué podría convenir ordenar por MAE en vez de R²?
2. Crea un solo modelo y míralo: `arbol = exp.create_model('dt')`. Compáralo con el mejor modelo del leaderboard: ¿cuánto se pierde al usar el modelo más simple?
3. Mira el gráfico de **error de predicción**: `exp.plot_model(mejor, plot='error')`. ¿El modelo acierta más en casas baratas o caras?
4. Compara `leaderboard` con `leaderboard_sin_outliers`: ¿la remoción automática de outliers cambió el ranking, las métricas o el modelo ganador? ¿Te parece una decisión defendible para este negocio?
5. Revisa `metricas_por_comuna`: ¿hay comunas donde el modelo se equivoque más? ¿Qué hipótesis de negocio podrían explicarlo?
6. En una frase: si tuvieras que **explicarle a la gerencia** por qué elegiste este modelo, ¿qué dirías sobre su error en UF y sus límites?


---

**Síntesis.** Predecir un número es **regresión**. Comparamos familias de modelos (lineal, árboles, ensambles) con un *leaderboard* basado en **validación cruzada**, elegimos por **métricas leídas en UF** (MAE/RMSE) más el R², probamos el efecto de `remove_outliers=True`, miramos el modelo por dentro (**residuos** e **importancia**) y lo confirmamos en el **conjunto de prueba** con tabla y gráfico real vs. predicho. Lo esencial no es el código, sino **interpretar y decidir** — y comunicar los **límites** del modelo.

Repasa con la **guía de estudio** y las **tarjetas** de la clase.
